# Aufgabe 1b – CNN zur Autoerkennung (ohne keras.models/keras.layers)


Hier erfolgt die Implementierung von CNN aus Aufgabe 1a nochmal komplett ohne die Keras-Schichten zu benutzen.
Das Laden der Daten und das gespeicherte Modell aus 1a, wird dafür allerdings wiederverwendet.




## Bibliotheken

Keras wird hierfür nur zum Laden der Daten wie in 1a verwendet, allerdings nciht für die Implementierung des CNNs

Numpy wird verwendet für:
- Bilder als Zahlenmatrizen zu laden
- Gewichte als Zufallszahlen zu initialisieren (np.random.randn)
- Den Vorwärts- und Rückwärtsdurchlauf manuell durch Matrix-Operationen zu berechnen




In [29]:
# numpy reicht für alles – keras nur zum Laden der Daten wie in 1a
import numpy as np
from keras.datasets import cifar10

print('numpy:', np.__version__)

numpy: 2.0.2


In [5]:
# CIFAR-10: 10 Klassen
(X_train_full, y_train_full), (X_test_full, y_test_full) = cifar10.load_data()

print('Trainingsdaten:', X_train_full.shape)
print('Testdaten:', X_test_full.shape)

Trainingsdaten: (50000, 32, 32, 3)
Testdaten: (10000, 32, 32, 3)


## Datenvorbereitung
Da nun ohne die Open-Source-Bibliotheken *keras* die Implementierung des CNNs erfolgt, wird der ganze Prozess viel langsamer andauern. Alle Alogrithmen werden in Python-Schleifen ohne GPU-Beschleunigung laufen.

Damit eine praktikable Trainingsdauer, aber dennoch ein aussagekräftiges Ergebnis erzielt wird, wurden die Traningsdaten auf 2000 und die Testdaten auf 400 reduziert.




In [6]:
N_TRAIN = 2000
N_TEST  = 400

# Normalisierung auf [0, 1] – gleicher Grund wie in 1a
# (ohne das explodieren die Gradienten sofort)
X_train = X_train_full[:N_TRAIN].astype(np.float32) / 255.0
y_train = (y_train_full[:N_TRAIN] == 1).astype(np.float32).flatten()

X_test  = X_test_full[:N_TEST].astype(np.float32) / 255.0
y_test  = (y_test_full[:N_TEST] == 1).astype(np.float32).flatten()

print(f'Training: {X_train.shape}, davon Autos: {int(y_train.sum())}')
print(f'Test:     {X_test.shape},  davon Autos: {int(y_test.sum())}')

Training: (2000, 32, 32, 3), davon Autos: 191
Test:     (400, 32, 32, 3),  davon Autos: 33


## Aktivierungsfunktionen

Wie in Aufgabe 1a werden die Aktivierungsfunktionen Sigmoid und ReLU verwendet. Diese Funktionen sind für die Nichtlinearität des Netzwerk wichtig. Ohne diese Funktionen wäre das gesamte CNN nur eine große lineare Gleichung und könnte keine komplexen Muster wie Autos erkennen.



ReLU: `max(0, x)` Es ist schneller für Hidden Layer und vermeidet das Vanishing Gradient Problem


Sigmoid: `S(x) = 1 / (1 + e^-x)`
Es erfolgt eine  Ausgabe zwischen 0 und 1, was als Wahrscheinlichkeit interpretieren wird. Da eine eine binäre Klassifikation (Auto vs. kein Auto) durchführen wird, ist Sigmoid ideal.

Eine Besonderheit der Sigmoid-Funktion ist, dass ihre Ableitung leicht aus dem Funktionswert berechnet werden kann: `s′(x)=s(x)⋅(1−s(x))`.


In [7]:
def sigmoid(x):
    # np.clip verhindert Overflow (Numerical Instability) bei sehr großen Werten
    x = np.clip(x, -500, 500)
    return 1.0 / (1.0 + np.exp(-x))

def sigmoid_ableitung(x):
    # Die Ableitung wird für die Delta-Regel benötigt.
    s = sigmoid(x)
    return s * (1 - s)

def relu(x):
    return np.maximum(0, x)

def relu_ableitung(x):
    # Der Gradient ist entweder 1 (Signal wird durchgereicht) oder 0 (Signal blockiert)
    return (x > 0).astype(np.float32)

# kurzer Test, ob es stimmt
print(f'sigmoid(0)  = {sigmoid(0):.4f} (wahres Ergbenis: 0.5)')
print(f'relu(-2)    = {relu(-2):.4f}  (wahres Ergebnis: 0.0)')
print(f'relu(5)     = {relu(5):.4f}  -(wahres Ergebnis: 5.0)')

sigmoid(0)  = 0.5000 (wahres Ergbenis: 0.5)
relu(-2)    = 0.0000  (wahres Ergebnis: 0.0
relu(5)     = 5.0000  -(wahres Ergebnis: 5.0


## Kostenfunktion – Binary Crossentropy

Das Ziel der Kostenfunktion ist es die Kosten C zu minimieren. Denn je kleiner C, desto sicherer und korrekter sind die Vorhersagen des Modells.

Für binäre Klassifikation (Auto=1, kein Auto=0):

`C = -[ y * log(ŷ) + (1-y) * log(1-ŷ) ]`

Die Ableitung davon wird dann für die Backprop gebraucht.

In [8]:
def binary_crossentropy(y_wahr, y_pred):
    # Epsilon-Clipping': In der Mathematik ist log(0) nicht definiert.
    # log(0) wäre minus-unendlich und das macht alles kaputt
    eps = 1e-12
    y_pred = np.clip(y_pred, eps, 1 - eps)

    # Die Summe der Log-Losses wird über den Batch gemittelt (np.mean),
    # um die Kosten unabhängig von der Batch-Größe vergleichbar zu machen
    return -np.mean(y_wahr * np.log(y_pred) + (1 - y_wahr) * np.log(1 - y_pred))

print('Kostenfunktion bereit.')

Kostenfunktion bereit.


# CNN-Schichten selbst implementieren

## Faltungsschicht (Conv2D)

Ein Neuron in einer Faltungsschicht ist nicht mit allen Pixeln des Eingabebildes verbunden, sondern nur mit einem kleinen Ausschnitt (dem Kernel/Filter). Dadurch lernt es lokale Merkmale wie Kanten, Ecken oder Rundungen.
Ein Filter/Kernel wird über das Bild geschoben und berechnet an jeder Position eine gewichtete Summe.
Mit `padding='same'` bleibt die Bildgröße erhalten, ohne dies würde ein 3x3-Kernel ein 32x32-Bild auf 30x30 verkleinern.

In [9]:
def conv2d_vorwaerts(bild, kernel, bias):
    """
    Vorwärtsdurchlauf der Faltungsschicht.
    bild:   (H, W, C_in)
    kernel: (kH, kW, C_in, C_out)
    bias:   (C_out,)
    Ausgabe: (H, W, C_out)  [padding=same]
    """
    H, W, C_in       = bild.shape
    kH, kW, _, C_out = kernel.shape

    # Padding so berechnen dass Ausgabe gleich groß wie Eingabe ist
    pad_h = kH // 2
    pad_w = kW // 2

    # Nullen um das Bild herum (nur H, W und nicht die Kanäle)
    bild_gepaddet = np.pad(bild, ((pad_h, pad_h), (pad_w, pad_w), (0, 0)), mode='constant')
    ausgabe = np.zeros((H, W, C_out), dtype=np.float32)

    # Sliding Window: Scannen des Merkmalsraums nach Mustern
    # Für jeden Filter und jede Position im Bild: Ausschnitt * Kernel summieren + Bias addieren
    # = genau das gleiche wie ein einzelnes Neuron (y = X*w + b), nur räumlich
    for c_out in range(C_out):
        for i in range(H):
            for j in range(W):
                ausschnitt = bild_gepaddet[i:i+kH, j:j+kW, :]
                ausgabe[i, j, c_out] = np.sum(ausschnitt * kernel[:, :, :, c_out]) + bias[c_out]

    return ausgabe


def conv2d_rueckwaerts(bild, kernel, grad_ausgabe):
    """
    Backpropagation durch die Faltungsschicht.
    Gibt Gradienten für kernel, bias und bild zurück.
    """
    H, W, C_in       = bild.shape
    kH, kW, _, C_out = kernel.shape
    pad_h = kH // 2
    pad_w = kW // 2

    bild_gepaddet = np.pad(bild, ((pad_h, pad_h), (pad_w, pad_w), (0, 0)), mode='constant')

    grad_kernel = np.zeros_like(kernel)
    grad_bias   = np.zeros(C_out, dtype=np.float32)
    grad_bild   = np.zeros_like(bild_gepaddet)

    # Backpropagation für Faltungen
    for c_out in range(C_out):
        for i in range(H):
            for j in range(W):
                ausschnitt = bild_gepaddet[i:i+kH, j:j+kW, :]
                g = grad_ausgabe[i, j, c_out]

                # += weil der gleiche Kernel an mehreren Positionen verwendet wird
                # weight sharing
                grad_kernel[:, :, :, c_out]   += g * ausschnitt
                grad_bias[c_out]              += g
                grad_bild[i:i+kH, j:j+kW, :] += g * kernel[:, :, :, c_out]

    # Padding wieder entfernen
    grad_bild_clean = grad_bild[pad_h:pad_h+H, pad_w:pad_w+W, :]
    return grad_kernel, grad_bias, grad_bild_clean


print('Conv2D-Schicht definiert.')

Conv2D-Schicht definiert.


## MaxPooling

MaxPooling wird verwendet um die Datenmenge zu reduzieren und das Netzwerk robuster zu machen.

Teilt das Bild in 2x2-Fenster auf und behält jeweils nur den größten Wert.
Ergebnis: Bildgröße halbiert sich (32x32 → 16x16) und es ergibt sich weniger Parameter, mehr Translationsinvarianz.

Im Rückwärtsdurchlauf geht der Gradient nur an die Stelle wo das Maximum war; alle anderen bekommen 0.

In [10]:
def maxpool_vorwaerts(bild, poolgroesse=2):
    """
    MaxPooling: 2x2-Fenster, Ausgabe halb so groß wie Eingabe.
    Gibt auch die Maske zurück und wird für Backprop benötigt.
    """
    H, W, C = bild.shape
    p = poolgroesse
    H_out = H // p
    W_out = W // p

    ausgabe = np.zeros((H_out, W_out, C), dtype=np.float32)
    masken  = np.zeros_like(bild, dtype=bool)

    for c in range(C):
        for i in range(H_out):
            for j in range(W_out):
                fenster = bild[i*p:(i+1)*p, j*p:(j+1)*p, c]
                maxwert = np.max(fenster)
                ausgabe[i, j, c] = maxwert
                # merken wo der Maximalwert war
                masken[i*p:(i+1)*p, j*p:(j+1)*p, c] = (fenster == maxwert)

    return ausgabe, masken


def maxpool_rueckwaerts(grad_ausgabe, masken, poolgroesse=2):
    """
    Backpropagation durch MaxPooling.
    Gradient wird nur an die Max-Position weitergegeben, Rest bekommt 0.
    """
    p = poolgroesse
    H_out, W_out, C = grad_ausgabe.shape
    grad_bild = np.zeros_like(masken, dtype=np.float32)

    for c in range(C):
        for i in range(H_out):
            for j in range(W_out):
                g = grad_ausgabe[i, j, c]
                grad_bild[i*p:(i+1)*p, j*p:(j+1)*p, c] = g * masken[i*p:(i+1)*p, j*p:(j+1)*p, c]

    return grad_bild


print('MaxPooling definiert.')

MaxPooling definiert.


## Vollverbundene Schicht (Dense Layer)

Die Dense Layer bildet den Abschluss des Netzwerks und führt die eigentliche Klassifikation durch. Sie basiert auf dem klassischen Neuronenmodell:

- Mathematische Operation: Die Schicht berechnet die gewichtete Summe der Eingaben plus einen Bias-Term `(z=X⋅W+b)` und wendet darauf die Aktivierungsfunktion an.

- Zweistufige Rückgabe: Um die Backpropagation korrekt durchzuführen, wird der Rechenschritt in den linearen Teil (z, den sogenannten Netzinput) und den nichtlinearen Teil (a, die Aktivierung) getrennt.

In [11]:
def dense_vorwaerts(eingabe, gewichte, bias, aktivierung='relu'):
    """
    Vollverbundene Schicht: z = X @ W + b, dann Aktivierungsfunktion.
    Gibt a (nach Aktivierung) UND z (vor Aktivierung) zurück.
    z wird für Backprop gebraucht.
    """
    z = np.dot(eingabe, gewichte) + bias

    if aktivierung == 'relu':
        a = relu(z)
    elif aktivierung == 'sigmoid':
        a = sigmoid(z)
    else:
        a = z  # keine Aktivierung

    return a, z


def dense_rueckwaerts(grad_a, z, eingabe, gewichte, aktivierung='relu'):
    """
    Backpropagation durch Dense Layer.
    Formel aus der Vorlesung: Δw_12 = ε * δ_2 * o_1
    (Lernrate * Fehlersignal * Ausgang vorherige Schicht)
    """
    # Kettenregel: dL/dz = dL/da * da/dz
    if aktivierung == 'relu':
        delta = grad_a * relu_ableitung(z)
    elif aktivierung == 'sigmoid':
        delta = grad_a * sigmoid_ableitung(z)
    else:
        delta = grad_a

    # Gradienten für Gewichte und Bias berechnen
    grad_gewichte = np.outer(eingabe, delta)   # dL/dW = x^T * delta
    grad_bias     = delta                      # dL/db = delta
    grad_eingabe  = np.dot(gewichte, delta)    # geht an vorherige Schicht weiter

    return delta, grad_gewichte, grad_bias, grad_eingabe


print('Dense Layer definiert.')

Dense Layer definiert.


## Gewichte initialisieren

Architektur (analog zu Aufgabe 1a, aber kleiner weil ohne GPU viel langsamer):
- **Conv**: 8 Filter, 3×3 Kernel, ReLU → MaxPooling (2×2)
- **Dense**: 32 Neuronen, ReLU
- **Ausgabe**: 1 Neuron, Sigmoid

Bildgröße nach den Schichten:
32×32×3 → Conv(same) → 32×32×8 → MaxPool → 16×16×8 → Flatten → 2048

In [12]:
# Seed damit Ergebnisse reproduzierbar sind
np.random.seed(42)

# Hyperparameter
LERNRATE   = 0.01   # zu groß: instabil, zu klein: ewig langsam
EPOCHEN    = 8
BATCH_SIZE = 64

N_FILTER   = 8     # Anzahl Convolutional Filter
KERN_SIZE  = 3     # 3x3 Kernel
N_DENSE    = 32    # Neuronen im Hidden Layer

# nach Conv(same) & MaxPool(2):
# 32x32x8 -> 16x16x8 -> flatten -> 2048
flat_dim = 16 * 16 * N_FILTER  # = 2048

# Zufällige kleine Gewichte (0.1 Skalierung)
# Alle Nullen wäre schlecht, denn dann würden alle Neuronen das Gleiche lernen
W_conv  = np.random.randn(KERN_SIZE, KERN_SIZE, 3, N_FILTER).astype(np.float32) * 0.1
b_conv  = np.zeros(N_FILTER, dtype=np.float32)

W_dense = np.random.randn(flat_dim, N_DENSE).astype(np.float32) * 0.1
b_dense = np.zeros(N_DENSE, dtype=np.float32)

W_out   = np.random.randn(N_DENSE, 1).astype(np.float32) * 0.1
b_out   = np.zeros(1, dtype=np.float32)

print('Architektur:')
print(f'  Conv:    {W_conv.shape}  ({N_FILTER} Filter)')
print(f'  MaxPool: 32x32x{N_FILTER} -> 16x16x{N_FILTER}')
print(f'  Flatten: {flat_dim}')
print(f'  Dense:   {flat_dim} -> {N_DENSE} (ReLU)')
print(f'  Ausgabe: {N_DENSE} -> 1 (Sigmoid)')
print(f'\nLernrate={LERNRATE}, Epochen={EPOCHEN}, Batchgröße={BATCH_SIZE}')

Architektur:
  Conv:    (3, 3, 3, 8)  (8 Filter)
  MaxPool: 32x32x8 -> 16x16x8
  Flatten: 2048
  Dense:   2048 -> 32 (ReLU)
  Ausgabe: 32 -> 1 (Sigmoid)

Lernrate=0.01, Epochen=8, Batchgröße=64


## Forward Pass

Der Forward Pass beschreibt den Weg der Bilddaten durch das Netzwerk – von den rohen Pixelwerten am Eingang bis zur finalen Klassenvorhersage am Ausgang.


Um die Gradienten zu berechnen, benötigt der Computer die Aktivierungswerte, die während des Vorwärtsdurchlaufs entstanden sind.
Anstatt allerdings diese Werte während des Rückwärtsdurchlaufs zeitaufwendig neu zu berechnen, werden sie in einem cache-Dictionary gespeichert

In [13]:
def vorwaerts(bild, W_conv, b_conv, W_dense, b_dense, W_out, b_out):
    """
    Vorwärtsdurchlauf: Bild -> Vorhersage
    Rückgabe: (vorhersage, cache)
    cache enthält alle Zwischenwerte für Backprop.
    """
    # 1) Faltungsschicht + ReLU
    conv_z = conv2d_vorwaerts(bild, W_conv, b_conv)
    conv_a = relu(conv_z)

    # 2) MaxPooling
    pool_a, pool_masken = maxpool_vorwaerts(conv_a, poolgroesse=2)

    # 3) Flatten: 3D -> 1D für Dense Layer
    flat = pool_a.flatten()

    # 4) Dense + ReLU
    dense_a, dense_z = dense_vorwaerts(flat, W_dense, b_dense, aktivierung='relu')

    # 5) Ausgabe-Neuron + Sigmoid -> Wahrscheinlichkeit
    out_a, out_z = dense_vorwaerts(dense_a, W_out, b_out, aktivierung='sigmoid')
    vorhersage = out_a[0]  # aus (1,) -> Skalar

    cache = {
        'bild': bild,
        'conv_z': conv_z, 'conv_a': conv_a,
        'pool_a': pool_a, 'pool_masken': pool_masken,
        'flat': flat,
        'dense_a': dense_a, 'dense_z': dense_z,
        'out_z': out_z
    }
    return vorhersage, cache


# kurzer Test – Wert sollte zwischen 0 und 1 liegen
testpred, _ = vorwaerts(X_train[0], W_conv, b_conv, W_dense, b_dense, W_out, b_out)
print(f'Test-Vorhersage: {testpred:.4f}  (muss zwischen 0 und 1 sein)')

Test-Vorhersage: 0.4639  (muss zwischen 0 und 1 sein)


## Backpropagation

Nachdem der Fehler am Ausgang des Netzwerks mittels der Kostenfunktion berechnet wurde, muss dieser zurückverteilt werden. Die Backpropagation ist das Verfahren, mit dem berechnet wird, wie jedes einzelne Gewicht im Netz zum Gesamtfehler beigetragen hat.

Formel für Gewichtsanpassung: `Δw_12 = ε · δ_2 · o_1`

Reihenfolge ist genau umgekehrt zum Forward Pass:
Ausgabe → Dense (Sigmoid) → Dense (ReLU) → Unflatten → MaxPool → Conv

In [14]:
def rueckwaerts(y_wahr, vorhersage, cache, W_conv, W_dense, W_out):
    """
    Backpropagation: berechnet Gradienten für alle Gewichte.
    Startet am Ausgang und arbeitet sich rückwärts durch.
    """
    eps = 1e-12

    # Gradient der Binary Crossentropy am Ausgang
    # dC/d(ŷ) = (ŷ - y) / (ŷ * (1 - ŷ))
    pred = np.clip(vorhersage, eps, 1 - eps)
    grad_vorhersage = np.array([(pred - y_wahr) / (pred * (1 - pred))])

    # Ausgabe-Neuron (Sigmoid) rückwärts
    delta_out, gW_out, gb_out, grad_dense_a = dense_rueckwaerts(
        grad_vorhersage, cache['out_z'], cache['dense_a'], W_out, aktivierung='sigmoid'
    )

    # Hidden Dense Layer (ReLU) rückwärts
    _, gW_dense, gb_dense, grad_flat = dense_rueckwaerts(
        grad_dense_a, cache['dense_z'], cache['flat'], W_dense, aktivierung='relu'
    )

    # Flatten rückgängig machen
    grad_pool_a = grad_flat.reshape(cache['pool_a'].shape)

    # MaxPooling rückwärts (Gradient nur an Max-Position)
    grad_conv_a = maxpool_rueckwaerts(grad_pool_a, cache['pool_masken'], poolgroesse=2)

    # ReLU nach Conv rückwärts
    # Gradient wird nur weitergeleitet wo der Vorwärtswert > 0 war
    grad_conv_z = grad_conv_a * relu_ableitung(cache['conv_z'])

    # Faltungsschicht rückwärts
    gW_conv, gb_conv, _ = conv2d_rueckwaerts(cache['bild'], W_conv, grad_conv_z)

    return {
        'W_conv':  gW_conv,  'b_conv':  gb_conv,
        'W_dense': gW_dense, 'b_dense': gb_dense,
        'W_out':   gW_out,   'b_out':   gb_out,
    }


print('Backpropagation definiert.')

Backpropagation definiert.


## Training

In diesem Abschnitt erfolgt der Lernalogrithmus. Ziel ist es, die Gewichte des Netzwerks so anzupassen, dass die Kostenfunktion minimiert wird. Es wird der Mini-Batch-Ansatz, der einen guten Ausgleich zwischen Schnelligkeit und Stabilität darstellt.

Wichtige Parameter für das Training
- **Epoche**: 1× alle Trainingsdaten durchlaufen
- **Batch**: Teilmenge der Daten (nicht einzeln, nicht alle auf einmal)
- **Lernrate ε**: Schrittgröße in Richtung des Gradienten

Gewichtsupdate: `w_neu = w_alt - ε * dC/dw`

In [15]:
verlauf_loss     = []
verlauf_val_loss = []
verlauf_acc      = []
verlauf_val_acc  = []

# 10% als Validierungsset abzweigen
split    = int(0.9 * len(X_train))
X_val, y_val = X_train[split:], y_train[split:]
X_tr,  y_tr  = X_train[:split],  y_train[:split]

print(f'Training: {len(X_tr)} Bilder | Validierung: {len(X_val)} Bilder')
print(f'{"Epoche":>7} | {"Train-Loss":>10} | {"Val-Loss":>10} | {"Train-Acc":>10} | {"Val-Acc":>10}')
print('-' * 62)

for epoche in range(EPOCHEN):

    # Daten am Anfang jeder Epoche mischen (wichtig für SGD)
    perm       = np.random.permutation(len(X_tr))
    X_gemischt = X_tr[perm]
    y_gemischt = y_tr[perm]

    # Mini-Batch Training
    for start in range(0, len(X_tr), BATCH_SIZE):
        ende    = min(start + BATCH_SIZE, len(X_tr))
        bX      = X_gemischt[start:ende]
        by      = y_gemischt[start:ende]

        # Gradienten für den gesamten Batch aufsummieren
        grad_summe = {
            'W_conv':  np.zeros_like(W_conv),
            'b_conv':  np.zeros_like(b_conv),
            'W_dense': np.zeros_like(W_dense),
            'b_dense': np.zeros_like(b_dense),
            'W_out':   np.zeros_like(W_out),
            'b_out':   np.zeros_like(b_out),
        }
        n = len(bX)

        for bild, label in zip(bX, by):
            pred, cache = vorwaerts(bild, W_conv, b_conv, W_dense, b_dense, W_out, b_out)
            grads = rueckwaerts(label, pred, cache, W_conv, W_dense, W_out)
            for key in grad_summe:
                grad_summe[key] += grads[key]

        # Gewichte aktualisieren: w = w - ε * Durchschnitts-Gradient
        W_conv  -= LERNRATE * grad_summe['W_conv']  / n
        b_conv  -= LERNRATE * grad_summe['b_conv']  / n
        W_dense -= LERNRATE * grad_summe['W_dense'] / n
        b_dense -= LERNRATE * grad_summe['b_dense'] / n
        W_out   -= LERNRATE * grad_summe['W_out']   / n
        b_out   -= LERNRATE * grad_summe['b_out']   / n

    # Auswertung nach jeder Epoche
    p_train = np.array([vorwaerts(x, W_conv, b_conv, W_dense, b_dense, W_out, b_out)[0] for x in X_tr])
    p_val   = np.array([vorwaerts(x, W_conv, b_conv, W_dense, b_dense, W_out, b_out)[0] for x in X_val])

    lt = binary_crossentropy(y_tr,  p_train)
    lv = binary_crossentropy(y_val, p_val)
    at = np.mean((p_train > 0.5) == y_tr)
    av = np.mean((p_val   > 0.5) == y_val)

    verlauf_loss.append(lt)
    verlauf_val_loss.append(lv)
    verlauf_acc.append(at)
    verlauf_val_acc.append(av)

    print(f'{epoche+1:>7} | {lt:>10.4f} | {lv:>10.4f} | {at*100:>9.2f}% | {av*100:>9.2f}%')

print('\nTraining fertig')

Training: 1800 Bilder | Validierung: 200 Bilder
 Epoche | Train-Loss |   Val-Loss |  Train-Acc |    Val-Acc
--------------------------------------------------------------
      1 |     0.3405 |     0.2301 |     89.78% |     96.50%
      2 |     0.3356 |     0.1965 |     89.78% |     96.50%
      3 |     0.3344 |     0.2185 |     89.78% |     96.50%
      4 |     0.3311 |     0.2102 |     89.78% |     96.50%
      5 |     0.3283 |     0.1947 |     89.78% |     96.50%
      6 |     0.3263 |     0.1988 |     89.78% |     96.50%
      7 |     0.3283 |     0.2146 |     89.78% |     96.50%
      8 |     0.3225 |     0.1898 |     89.78% |     96.50%

Training fertig


In [23]:
# Gewichte in einer npz-Datei speichern
np.savez('cardetector_scratch.npz',
         W_conv=W_conv,   b_conv=b_conv,
         W_dense=W_dense, b_dense=b_dense,
         W_out=W_out,     b_out=b_out)

print('Gespeichert: cardetector.npz')

Gespeichert: cardetector.npz
